# 3. Advanced Add-on Exercises: Graphs & Agents

This notebook contains **standalone advanced add-on exercises** that build on the ideas from `ex1_run_graph.ipynb` and `ex2_create_my_graph.ipynb`, but can be approached independently once you are familiar with:

- The repository's multi-agent graph (`src/graph.py`).
- The shared `State` model and how it is updated by different nodes.
- Basic agent construction and tools (from `ex2_create_my_graph.ipynb`).

The goal is to:
- Better understand how the graph and agents behave across different runs.
- Extend the behavior of the existing agents beyond what you did in Exercise 2.


In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Construct src-path and append it to sys.path
src_path = Path.cwd().parent.parent
sys.path.append(str(src_path))

# Initialize settings-object that loads the OpenAI API and Tavily API keys + langfuse credentials to the environment
from core.settings import settings  # noqa: F401


## 3.a Add-on Exercise: Compare Graph Runs & Agent Behaviour

In earlier exercises, you ran the multi-agent graph once and inspected the final answer and plan.

In this add-on exercise, you will:
- Wrap a single graph run into a **reusable helper** that returns a structured summary.
- Use it to run **several different queries** and compare how the plan and the agents' behaviour change.

This helps you reason about:
- How the `PlannerNode` structures plans for different questions.
- How often replanning happens.
- Which agents contribute most to the conversation.


In [ ]:
from collections import Counter
from typing import Dict, List, Any

from agents.planner import PlannerNode
from graph import graph
from models.agents import Agents
from models.state import Message, State

async def run_graph_with_summary(user_query: str, enabled_agents: List[Agents]) -> Dict[str, Any]:
    """Run the multi-agent graph once and return a structured summary.

    TODO: Implement this helper. It should:
    1. Build a fresh `State` instance from the given `user_query` and `enabled_agents`.
       - Use `messages=[Message(content=user_query)]` as a minimal starting history.
    2. Run the graph starting from `PlannerNode()` via `await graph.run(...)`.
    3. Inspect the final state (plan, messages, replans) and return a dictionary, e.g.:
       {
         "user_query": ...,
         "plan_type": ...,                # e.g. 'initial' or 'replan'
         "num_plan_steps": ...,
         "num_replans_total": ...,        # derive from `state.replan_attempts`
         "messages_per_agent": {...},     # counts keyed by `creator`
         "final_answer": ...,
       }

    You are free to add more fields if they help your analysis.
    """
    raise NotImplementedError("Implement `run_graph_with_summary` in this exercise.")


### 3.a Task: Run and Compare Multiple Queries

After you implement `run_graph_with_summary`, use it to run at least **three** different electricity-market-related queries, for example:

- A simple factual question (e.g. about a single country's electricity prices today).
- A comparative question (e.g. compare two countries or time periods).
- A more open-ended analytical question (e.g. drivers of price volatility).

For each run, compare:
- The plan type and number of steps.
- How often replanning occurred.
- How many messages each agent contributed.

Use the code cell below as a starting point and adapt it as you like.


In [ ]:
# TODO: After implementing `run_graph_with_summary`, uncomment and adapt this cell.

# example_queries = [
#     "What are the main drivers of day-ahead electricity prices in Germany today?",
#     "Compare day-ahead electricity price trends between Germany and France over the last week.",
#     "How does increasing solar generation share typically affect intraday price volatility in Germany?",
# ]
#
# enabled = [Agents.WEB_RESEARCHER, Agents.SYNTHESIZER]
#
# summaries = []
# for q in example_queries:
#     summary = await run_graph_with_summary(q, enabled)
#     summaries.append(summary)
#
# for s in summaries:
#     print("=== Query ===")
#     print(s["user_query"])
#     print("Plan type:", s.get("plan_type"))
#     print("# steps:", s.get("num_plan_steps"))
#     print("# replans total:", s.get("num_replans_total"))
#     print("Messages per agent:", s.get("messages_per_agent"))
#     print()


## 3.b Add-on Exercise: Extend the Agent Setup Beyond Exercise 2

In `ex2_create_my_graph.ipynb`, you created a specialized visualization agent and integrated it into a graph.

In this add-on exercise, you will create a **manual two-stage pipeline** that composes existing agents **without** using `pydantic-graph` directly:

1. A `WebResearcher` step that gathers raw information from the web.
2. A `Synthesizer` step that reads the accumulated `State` and produces a structured answer.

This extends the agent behaviour beyond Exercise 2 by orchestrating multiple agents yourself using the shared `State` model.


In [ ]:
from typing import Tuple

from agents.web_researcher.web_researcher import web_researcher
from agents.synthesizer.synthesizer import synthesizer
from models.agents import Agents
from models.state import Message, State

async def research_and_synthesize(user_query: str) -> Tuple[str, State]:
    """Run a two-stage pipeline: Web Researcher -> Synthesizer.

    TODO: Implement this helper so that it:
    1. Creates a fresh `State` with the user's question as the first message.
       - e.g. `messages=[Message(content=user_query)]`, `user_query=user_query`,
         `enabled_agents=[Agents.WEB_RESEARCHER, Agents.SYNTHESIZER]`.
    2. Calls the `web_researcher` agent with the user's query and the state as `deps`.
       - Append the returned `Message` to `state.messages` and set `creator=Agents.WEB_RESEARCHER`.
    3. Constructs a suitable synthesizer prompt that tells the `synthesizer` agent to:
       - Read the accumulated `state.messages` (including web research results).
       - Produce a concise, well-structured answer for the user.
    4. Calls the `synthesizer` agent with this prompt and the same `state` as `deps`.
       - Append the returned `Message` to `state.messages` and set `creator=Agents.SYNTHESIZER`.
    5. Returns a tuple `(final_answer, state)`, where `final_answer` is the content of the synthesizer's output.
    """
    raise NotImplementedError("Implement `research_and_synthesize` in this exercise.")


### 3.b Task: Test and Inspect Your Two-Stage Pipeline

After you implement `research_and_synthesize`, run it for a few different queries, e.g.:

- Explain today's main drivers of electricity prices in Germany.
- Compare long-term electricity price trends between two European countries.

Inspect:
- The intermediate `Message` produced by the Web Researcher.
- How the Synthesizer transforms that into a final answer.
- How the `State` object evolves (messages, enabled agents, etc.).


In [ ]:
# TODO: After implementing `research_and_synthesize`, uncomment and adapt this cell.

# query = "Explain today's main drivers of electricity prices in Germany."
# final_answer, final_state = await research_and_synthesize(query)
#
# print("=== Final answer ===")
# print(final_answer)
#
# print("=== Messages in state (creator, first 120 chars) ===")
# for m in final_state.messages:
#     print(m.creator, '-', m.content[:120].replace('\n', ' '), '...')
